In [0]:
# =============================================================================
# CELL 1: IMPORTS & LOAD BASE TABLES
# =============================================================================

from pyspark.sql import functions as F
from pyspark.sql.functions import col, lit, when, broadcast

# --- Base Tables ---
df_snapshot = spark.table("cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot")
df_transactions = spark.table("cpx_dataanalytics_gold.customer360.c360_f_transaction")
df_ticket_type = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_ticket_type")

In [0]:
# =============================================================================
# VERIFY: release date column name in the film dimension
# =============================================================================
df_film_raw = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_film")
print([c for c in df_film_raw.columns if "date" in c.lower() or "release" in c.lower() or "open" in c.lower()])

In [0]:
# # =============================================================================
# # TOP-TIER AVG VISITS (P12M, all visits) — CC vs Never-CC
# # Cumulative top groups: top 10% / 25% / 50% by visits, avg within each.
# # =============================================================================
# from pyspark.sql.window import Window

# def top_tier_avgs(df, cohort_name):
#     # percentile rank: 0 = most visits, 1 = fewest
#     w = Window.orderBy(col("visits_12m").desc())
#     ranked = df.withColumn("pct_rank", F.percent_rank().over(w))
#     rows = {}
#     for label, cutoff in [("Top 10%", 0.10), ("Top 25%", 0.25), ("Top 50%", 0.50)]:
#         rows[label] = ranked.filter(col("pct_rank") < cutoff) \
#                             .agg(F.round(F.avg("visits_12m"), 1)).collect()[0][0]
#     rows["All members"] = df.agg(F.round(F.avg("visits_12m"), 1)).collect()[0][0]
#     print(f"\n{cohort_name}")
#     for k, v in rows.items():
#         print(f"  {k:<12}: {v} visits/yr")

# top_tier_avgs(df_cmp12_all.filter(col("cohort") == "Active CC Member"), "Active CC Member")
# top_tier_avgs(df_cmp12_all.filter(col("cohort") == "Never CC Member"), "Never CC Member")

In [0]:
# =============================================================================
# CELL 2: FILTER TO AUG 2021+ & AGGREGATE REVENUE/TICKETS BY CC STATUS
# =============================================================================
# A customer can have transactions BOTH as CC and non-CC (they may join/leave).
# We use F.when conditions to split revenue into CC vs non-CC buckets per CDE_ID.
# TransactionDateid format: YYYYMMDD integer
# =============================================================================

# Select only columns needed downstream & cache — avoids re-scanning the full table in cells 5-10
_txn_cols = ["CDE_ID", "TransactionDateid", "VisitDateId", "LocationId", "Net_Revenue",
             "Quantity", "TicketFLAG", "ConcessionFLAG", "CineClubFLAG",
             "CineClub_Discount_Type_ID", "TicketTypeSkey", "COTFLAG", "film_id"]
df_txn = (
    df_transactions
    .select(*_txn_cols)
    .filter(col("TransactionDateid") >= 20210801)
    .cache()
)

df_revenue = df_txn.groupBy("CDE_ID").agg(
    # --- Ticket Revenue ---
    F.sum(when((col("TicketFLAG") == "Yes") & (col("CineClubFLAG") == "Yes"), col("Net_Revenue")).otherwise(0)).alias("CC_Ticket_Revenue"),
    F.sum(when((col("TicketFLAG") == "Yes") & (col("CineClubFLAG") == "No"), col("Net_Revenue")).otherwise(0)).alias("NonCC_Ticket_Revenue"),
    # --- Concession Revenue ---
    F.sum(when((col("ConcessionFLAG") == "Yes") & (col("CineClubFLAG") == "Yes"), col("Net_Revenue")).otherwise(0)).alias("CC_Concession_Revenue"),
    F.sum(when((col("ConcessionFLAG") == "Yes") & (col("CineClubFLAG") == "No"), col("Net_Revenue")).otherwise(0)).alias("NonCC_Concession_Revenue"),
    # --- Ticket Counts ---
    F.sum(when((col("TicketFLAG") == "Yes") & (col("CineClubFLAG") == "Yes"), col("Quantity")).otherwise(0)).alias("CC_Tickets"),
    F.sum(when((col("TicketFLAG") == "Yes") & (col("CineClubFLAG") == "No"), col("Quantity")).otherwise(0)).alias("NonCC_Tickets")
)

In [0]:
# =============================================================================
# CELL 3: CLASSIFY CUSTOMERS INTO 3 COHORTS
# =============================================================================
# Cohort 1: "Current Active CC Member" — in latest snapshot with renewal_status LIKE 'Active%'
# Cohort 2: "Former CC Member"         — had CC transactions but NOT currently active
# Cohort 3: "Never CC Member"          — never had a single CC transaction
# =============================================================================

# Step 1: Get currently active CC members from latest snapshot
latest_snapshot = df_snapshot.agg(F.max("snapshot_date_id")).collect()[0][0]
print(f"Latest snapshot date: {latest_snapshot}")

df_active_cc = (
    df_snapshot
    .filter(col("snapshot_date_id") == latest_snapshot)
    .filter(col("renewal_status").like("Active%"))
    .select("cde_id")
    .distinct()
    .cache()
)

# Step 2: Get all CDE_IDs that EVER had a CineClub transaction
df_ever_cc = (
    df_txn
    .filter(col("CineClubFLAG") == "Yes")
    .select(col("CDE_ID").cast("string").alias("cde_id"))
    .distinct()
    .cache()
)

# Step 3: Join to revenue table and assign cohort labels
# (Rename keys before join to avoid column ambiguity)
df_active_cc_renamed = df_active_cc.select(col("cde_id").cast("int").alias("active_cde_id"))
df_ever_cc_renamed = df_ever_cc.select(col("cde_id").cast("int").alias("ever_cc_cde_id"))

df_revenue_status = (
    df_revenue
    .join(broadcast(df_active_cc_renamed), col("CDE_ID") == col("active_cde_id"), "left")
    .join(broadcast(df_ever_cc_renamed), col("CDE_ID") == col("ever_cc_cde_id"), "left")
    .withColumn(
        "Current_Status",
        when(col("active_cde_id").isNotNull(), lit("Current Active CC Member"))
        .when(col("ever_cc_cde_id").isNotNull(), lit("Former CC Member"))
        .otherwise(lit("Never CC Member"))
    )
    .drop("active_cde_id", "ever_cc_cde_id")
)

# Verify cohort counts
# df_revenue_status.groupBy("Current_Status").count().show()

In [0]:
# =============================================================================
# CELL 4: CURRENT ACTIVE CC MEMBER SAVINGS ("YOU HAVE SAVED")
# =============================================================================
# Logic (from CineClub Savings notebook):
#   - Ticket savings   = (discount type 5 & 6 qty) * (GA price - $9.99)
#   - Concession savings = 25% of discounted concession revenue (type 2)
#   - Gaming savings   = 25% of discounted gaming revenue (type 3)
#   - Online booking   = $1 per online ticket visit with CC discount
#   - Adult tickets    = report_code1 = 'AA', childflag = 'No', seniorflag = 'No'
# =============================================================================

%pip install openpyxl -q
import pandas as pd

# --- Step 1: Load GA Prices by Location & Year (from Excel) ---
xlsx_path = "/Volumes/data_exploration_hub/marketing_hub/offline_data/General Admit Prices.xlsx"
df_ga_xlsx = pd.read_excel(xlsx_path, sheet_name="Sheet3")

year_cols = [c for c in df_ga_xlsx.columns if c != "LocationID"]
df_ga_long = (
    df_ga_xlsx
    .melt(id_vars="LocationID", value_vars=year_cols, var_name="visit_year", value_name="ga_ticket_price")
    .rename(columns={"LocationID": "LocationId"})
)
df_ga_long["visit_year"] = df_ga_long["visit_year"].astype(int)
df_ga_prices = spark.createDataFrame(df_ga_long)

# --- Step 2: Build visit-level transactions for active CC members ---
df_cc_visits = (
    df_txn
    .filter(col("CineClubFLAG") == "Yes")
    # Only active CC members
    .join(broadcast(df_active_cc), df_txn["CDE_ID"] == df_active_cc["cde_id"], "inner")
    .drop(df_active_cc["cde_id"])
    # Join ticket type for adult ticket filter (report_code1)
    .join(broadcast(df_ticket_type.select("tickettypeskey", "report_code1", "childflag", "seniorflag")),
          df_txn["TicketTypeSkey"] == df_ticket_type["tickettypeskey"], "left")
    # Join GA prices by location and year
    .withColumn("visit_year", F.year(F.to_date(col("VisitDateId").cast("string"), "yyyyMMdd")))
    .join(broadcast(df_ga_prices), on=["LocationId", "visit_year"], how="left")
)

# --- Step 3: Aggregate savings per CDE_ID ---
df_cc_savings = (
    df_cc_visits
    .groupBy("CDE_ID")
    .agg(
        # Ticket savings: CC ticket price is $9.99, saving = (GA price - 9.99) per ticket
        F.round(F.sum(
            when(col("CineClub_Discount_Type_ID").isin(5, 6),
                 col("Quantity") * (F.coalesce(col("ga_ticket_price"), lit(9.99)) - lit(9.99))
            ).otherwise(0)
        ), 2).alias("ticket_savings"),

        # Concession savings: 20% off concessions (discount type 2)
        F.round(F.sum(
            when((col("ConcessionFLAG") == "Yes") & (col("CineClub_Discount_Type_ID") == 2),
                 col("Net_Revenue") * 0.25
            ).otherwise(0)
        ), 2).alias("concession_savings"),

        # Gaming savings: 20% off gaming/LBE (discount type 3)
        F.round(F.sum(
            when((col("ConcessionFLAG") == "Yes") & (col("CineClub_Discount_Type_ID") == 3),
                 col("Net_Revenue") * 0.25
            ).otherwise(0)
        ), 2).alias("gaming_savings"),

        # Online booking: $1 per online ticket booking with CC discount
        F.countDistinct(
            when((col("TicketFLAG") == "Yes") & (col("COTFLAG") == "Yes") &
                (col("CineClubFLAG") == "Yes"),
                    F.concat_ws("|", col("VisitDateId").cast("string"), col("LocationId").cast("string"))
                )).alias("online_booking_savings"),

        # Adult ticket count (report_code1 = AA, not child, not senior)
        F.sum(
            when((col("TicketFLAG") == "Yes") & (col("report_code1") == "AA") &
                 (col("childflag") == "No") & (col("seniorflag") == "No"),
                 col("Quantity")
            ).otherwise(0)
        ).cast("int").alias("CC_Adult_Tickets"),

        # Total ticket count (all types)
        F.sum(when(col("TicketFLAG") == "Yes", col("Quantity")).otherwise(0)
        ).cast("int").alias("CC_Total_Tickets")
    )
    # Total savings = sum of all components
    .withColumn("total_savings", F.round(
        col("ticket_savings") + col("concession_savings") + col("gaming_savings") + col("online_booking_savings"), 2
    ))
)

# print(f"Active CC Members with savings: {df_cc_savings.count():,}")
# display(df_cc_savings)

In [0]:
# =============================================================================
# CELL 5: NON-CC PROJECTED SAVINGS ("YOU COULD HAVE SAVED")
# =============================================================================
# Target: Everyone EXCEPT current active CC members (Former CC + Never CC)
# Only looks at their non-CC transactions (CineClubFLAG = 'No')
#
# Savings logic (simplified vs actual CC savings):
#   - Ticket savings    = actual revenue paid - (adult ticket count * $9.99), floored at 0
#   - Concession savings = 20% of concession revenue
#   - Adult tickets     = report_code1 = 'AA', childflag = 'No', seniorflag = 'No'
#   - Refund rows (Quantity <= 0) excluded from revenue, adult count, and total count
#     to prevent negative-qty returns from deflating total below adult.
#   - ticket_type dimension deduplicated on tickettypeskey to avoid join fan-out.
# =============================================================================

# --- Step 1: Get non-CC transactions, exclude active CC members ---
df_noncc_visits = (
    df_txn
    .filter(col("CineClubFLAG") == "No")
    .join(broadcast(df_active_cc), df_txn["CDE_ID"] == df_active_cc["cde_id"], "left_anti")
    # Join ticket type for adult filter
    .join(broadcast(df_ticket_type.select("tickettypeskey", "report_code1", "childflag", "seniorflag").dropDuplicates(["tickettypeskey"])),
          df_txn["TicketTypeSkey"] == df_ticket_type["tickettypeskey"], "left")
    # Tag as Former vs Never CC
    .join(broadcast(df_ever_cc.withColumnRenamed("cde_id", "_ever_cde_id")),
          col("CDE_ID") == col("_ever_cde_id"), "left")
    .withColumn("Current_Status",
                when(col("_ever_cde_id").isNotNull(), lit("Former CC Member"))
                .otherwise(lit("Never CC Member")))
    .drop("_ever_cde_id")
)

# --- Step 2: Aggregate per CDE_ID ---
df_noncc_savings = (
    df_noncc_visits
    .groupBy("CDE_ID", "Current_Status")
    .agg(
        # What they actually paid for adult tickets (positive qty only — excludes refunds)
        F.round(F.sum(
            when((col("TicketFLAG") == "Yes") & (col("report_code1") == "AA") &
                 (col("childflag") == "No") & (col("seniorflag") == "No"),
                 col("Net_Revenue")
            ).otherwise(0)
        ), 2).alias("actual_adult_ticket_revenue"),

        # Adult ticket count (positive quantities only — excludes refunds)
        F.sum(
            when((col("TicketFLAG") == "Yes") & (col("report_code1") == "AA") &
                 (col("childflag") == "No") & (col("seniorflag") == "No"),
                 col("Quantity")
            ).otherwise(0)
        ).cast("int").alias("NonCC_Adult_Tickets"),

        # Total concession revenue
        F.round(F.sum(
            when(col("ConcessionFLAG") == "Yes", col("Net_Revenue")).otherwise(0)
        ), 2).alias("concession_revenue"),

        # Total tickets (all types, positive quantities only — excludes refunds)
        F.sum(when((col("TicketFLAG") == "Yes") & (col("Quantity") > 0), col("Quantity")).otherwise(0)
        ).cast("int").alias("NonCC_Total_Tickets")
    )
)

# --- Step 3: Calculate projected savings ---
# Ticket: what they paid minus what CC price would have been ($9.99/ticket), floor at 0
df_noncc_savings = df_noncc_savings.withColumn(
    "ticket_savings",
    F.round((col("actual_adult_ticket_revenue") - (col("NonCC_Adult_Tickets") * lit(9.99))), 2)
    #F.round(F.greatest(col("actual_adult_ticket_revenue") - (col("NonCC_Adult_Tickets") * lit(9.99)), lit(0)), 2)
)
# Concession: 20% of what they spent
df_noncc_savings = df_noncc_savings.withColumn(
    "concession_savings",
    F.round(F.greatest(col("concession_revenue") * 0.20, lit(0)), 2)
)
# Total
df_noncc_savings = df_noncc_savings.withColumn(
    "total_could_have_saved",
    F.round(col("ticket_savings") + col("concession_savings"), 2)
)

# print(f"Non-CC Members with projected savings: {df_noncc_savings.count():,}")
# display(df_noncc_savings)

In [0]:
# =============================================================================
# CELL 6: FORMER CC MEMBER ACTUAL SAVINGS (WHILE THEY WERE CC)
# =============================================================================
# Same savings formula as Cell 4 but for members who are no longer active.
# Only looks at their CC transactions (CineClubFLAG = 'Yes').
# =============================================================================

# --- Step 1: Identify former CC members (ever CC minus currently active) ---
df_former_cc = df_ever_cc.join(df_active_cc, on="cde_id", how="left_anti")

# --- Step 2: Get their CC transactions ---
df_former_cc_visits = (
    df_txn
    .filter(col("CineClubFLAG") == "Yes")
    .join(broadcast(df_former_cc), df_txn["CDE_ID"] == df_former_cc["cde_id"], "inner")
    .drop(df_former_cc["cde_id"])
    .join(broadcast(df_ticket_type.select("tickettypeskey", "report_code1", "childflag", "seniorflag")),
          df_txn["TicketTypeSkey"] == df_ticket_type["tickettypeskey"], "left")
    .withColumn("visit_year", F.year(F.to_date(col("VisitDateId").cast("string"), "yyyyMMdd")))
    .join(broadcast(df_ga_prices), on=["LocationId", "visit_year"], how="left")
)

# --- Step 3: Aggregate savings (same logic as Cell 4) ---
df_former_cc_savings = (
    df_former_cc_visits
    .groupBy("CDE_ID")
    .agg(
        F.round(F.sum(
            when(col("CineClub_Discount_Type_ID").isin(5, 6),
                 col("Quantity") * (F.coalesce(col("ga_ticket_price"), lit(9.99)) - lit(9.99))
            ).otherwise(0)
        ), 2).alias("former_cc_ticket_savings"),

        F.round(F.sum(
            when((col("ConcessionFLAG") == "Yes") & (col("CineClub_Discount_Type_ID") == 2),
                 col("Net_Revenue") * 0.25
            ).otherwise(0)
        ), 2).alias("former_cc_concession_savings"),

        F.round(F.sum(
            when((col("ConcessionFLAG") == "Yes") & (col("CineClub_Discount_Type_ID") == 3),
                 col("Net_Revenue") * 0.25
            ).otherwise(0)
        ), 2).alias("former_cc_gaming_savings"),

        F.countDistinct(
            when((col("TicketFLAG") == "Yes") & (col("COTFLAG") == "Yes") &
                (col("CineClubFLAG") == "Yes"),
                F.concat_ws("|", col("VisitDateId").cast("string"), col("LocationId").cast("string"))
            )
        ).alias("former_cc_online_savings"),

        F.sum(
            when((col("TicketFLAG") == "Yes") & (col("report_code1") == "AA") &
                 (col("childflag") == "No") & (col("seniorflag") == "No"),
                 col("Quantity")
            ).otherwise(0)
        ).cast("int").alias("Former_CC_Adult_Tickets"),

        F.sum(when(col("TicketFLAG") == "Yes", col("Quantity")).otherwise(0)
        ).cast("int").alias("Former_CC_Total_Tickets")
    )
    .withColumn("former_cc_total_savings", F.round(
        col("former_cc_ticket_savings") + col("former_cc_concession_savings") +
        col("former_cc_gaming_savings") + col("former_cc_online_savings"), 2
    ))
)

# print(f"Former CC Members with savings: {df_former_cc_savings.count():,}")
# display(df_former_cc_savings)

In [0]:
# =============================================================================
# CELL 7: COMPILE FINAL OUTPUT TABLE
# =============================================================================
# Combines all 3 cohorts into one unified table.
# Excludes sentinel CDE_IDs (-1 and NULL) and floors all savings at $0.
#
# Output columns:
#   CDE_ID, Current_Status, CC_Adult_Tickets, CC_Total_Tickets,
#   Never_Adult_Tickets, Never_Total_Tickets,
#   cc_ticket_savings, cc_concession_savings, cc_gaming_savings,
#   cc_online_booking_savings, you_have_saved, you_could_have_saved
# =============================================================================

# Helper: filter out system sentinel CDE_IDs
def exclude_sentinels(df):
    return df.filter((col("CDE_ID").isNotNull()) & (col("CDE_ID") != -1))

# P12M visitor list — used to restrict Never CC Members to recent visitors only
p12m_cutoff = 20250805
df_p12m_visitors = (
    df_txn
    .filter(col("TransactionDateid") >= p12m_cutoff)
    .select("CDE_ID")
    .distinct()
)

# --- Cohort 1: Current Active CC Members ---
df_final_cc = (
    exclude_sentinels(df_cc_savings)
    .withColumn("Current_Status", lit("Current Active CC Member"))
    .withColumn("total_savings", F.greatest(col("total_savings"), lit(0)))
    .select(
        col("CDE_ID"),
        col("Current_Status"),
        col("CC_Adult_Tickets"),
        col("CC_Total_Tickets"),
        lit(0).cast("int").alias("Never_Adult_Tickets"),
        lit(0).cast("int").alias("Never_Total_Tickets"),
        col("ticket_savings").alias("cc_ticket_savings"),
        col("concession_savings").alias("cc_concession_savings"),
        col("gaming_savings").alias("cc_gaming_savings"),
        col("online_booking_savings").alias("cc_online_booking_savings"),
        col("total_savings").alias("you_have_saved"),
        lit(None).cast("double").alias("you_could_have_saved")
    )
)

# --- Cohort 2: Former CC Members (actual CC savings + projected non-CC savings) ---
df_former_projected = (
    df_noncc_savings
    .filter(col("Current_Status") == "Former CC Member")
    .select("CDE_ID", "total_could_have_saved", "NonCC_Adult_Tickets", "NonCC_Total_Tickets")
)

df_final_former = (
    exclude_sentinels(df_former_cc_savings)
    .join(df_former_projected, on="CDE_ID", how="left")
    .withColumn("Current_Status", lit("Former CC Member"))
    .withColumn("former_cc_total_savings", F.greatest(col("former_cc_total_savings"), lit(0)))
    .withColumn("total_could_have_saved", F.greatest(F.coalesce(col("total_could_have_saved"), lit(0)), lit(0)))
    .select(
        col("CDE_ID"),
        col("Current_Status"),
        col("Former_CC_Adult_Tickets").alias("CC_Adult_Tickets"),
        col("Former_CC_Total_Tickets").alias("CC_Total_Tickets"),
        F.coalesce(col("NonCC_Adult_Tickets"), lit(0)).cast("int").alias("Never_Adult_Tickets"),
        F.coalesce(col("NonCC_Total_Tickets"), lit(0)).cast("int").alias("Never_Total_Tickets"),
        col("former_cc_ticket_savings").alias("cc_ticket_savings"),
        col("former_cc_concession_savings").alias("cc_concession_savings"),
        col("former_cc_gaming_savings").alias("cc_gaming_savings"),
        col("former_cc_online_savings").alias("cc_online_booking_savings"),
        col("former_cc_total_savings").alias("you_have_saved"),
        col("total_could_have_saved").alias("you_could_have_saved")
    )
)

# --- Cohort 3: Never CC Members (projected savings only, P12M visitors only) ---
df_final_never = (
    exclude_sentinels(df_noncc_savings.filter(col("Current_Status") == "Never CC Member"))
    .join(df_p12m_visitors, on="CDE_ID", how="inner")  # restrict to last-12-months visitors
    .withColumn("total_could_have_saved", F.greatest(col("total_could_have_saved"), lit(0)))
    .select(
        col("CDE_ID"),
        col("Current_Status"),
        lit(0).cast("int").alias("CC_Adult_Tickets"),
        lit(0).cast("int").alias("CC_Total_Tickets"),
        col("NonCC_Adult_Tickets").cast("int").alias("Never_Adult_Tickets"),
        col("NonCC_Total_Tickets").cast("int").alias("Never_Total_Tickets"),
        lit(0.0).alias("cc_ticket_savings"),
        lit(0.0).alias("cc_concession_savings"),
        lit(0.0).alias("cc_gaming_savings"),
        lit(0.0).alias("cc_online_booking_savings"),
        lit(0.0).alias("you_have_saved"),
        col("total_could_have_saved").alias("you_could_have_saved")
    )
)

# --- Union all 3 cohorts ---
df_final = df_final_cc.unionByName(df_final_former).unionByName(df_final_never)

# # --- Print summary ---
# print("FINAL OUTPUT TABLE")
# print("=" * 60)
# df_final.groupBy("Current_Status").agg(
#     F.count("*").alias("count"),
#     F.round(F.avg("you_have_saved"), 2).alias("avg_you_have_saved"),
#     F.round(F.avg("you_could_have_saved"), 2).alias("avg_you_could_have_saved")
# ).show(truncate=False)

# print(f"Total rows: {df_final.count():,}")
# display(df_final)

In [0]:
# =============================================================================
# CELL 7c: ADD UNIQUE TITLES WATCHED (TOTAL & AS CC) TO df_final
# =============================================================================
# For every CDE_ID in df_final, count distinct title_ids they watched:
#   1. unique_titles_total — all ticketed watches (CC + non-CC)
#   2. unique_titles_cc    — ticketed watches while CineClubFLAG = 'Yes'
# "Watched" = TicketFLAG = 'Yes'. Same Aug 2021+ window as the rest of the notebook.
# =============================================================================

# Film dimension: film_id -> title_id (dedupe on film_id)
df_film = (
    spark.table("cpx_dataanalytics_gold.customer360.cpx_d_film")
    .select("film_id", "title_id")
    .dropDuplicates(["film_id"])
)

# Ticketed transactions since Aug 2021, enriched with title_id
df_watched = (
    df_txn
    .filter(col("TicketFLAG") == "Yes")
    .join(broadcast(df_film), on="film_id", how="left")
    .select("CDE_ID", "title_id", "CineClubFLAG")
)

# Count distinct titles per member, total vs as-CC
df_unique_titles = df_watched.groupBy("CDE_ID").agg(
    F.countDistinct("title_id").alias("unique_titles_total"),
    F.countDistinct(
        when(col("CineClubFLAG") == "Yes", col("title_id"))
    ).alias("unique_titles_cc"),
)

# Attach to df_final (left join keeps all members; fill non-watchers with 0)
df_final = (
    df_final
    .join(df_unique_titles, on="CDE_ID", how="left")
    .fillna(0, subset=["unique_titles_total", "unique_titles_cc"])
)

# df_final.select(
#     "CDE_ID", "Current_Status", "unique_titles_total", "unique_titles_cc"
# ).show(10, truncate=False)

In [0]:
# Replace "Current Active CC Member" in Current_Status with the member's plan_id
df_plan_id = (
    df_snapshot
    .filter(col("snapshot_date_id") == latest_snapshot)
    .filter(col("renewal_status").like("Active%"))
    .select(col("cde_id").cast("int").alias("plan_cde_id"), col("plan_id").alias("snapshot_plan_id"))
    .dropDuplicates(["plan_cde_id"])
)

df_final = (
    df_final
    .join(broadcast(df_plan_id), col("CDE_ID") == col("plan_cde_id"), "left")
    .drop("plan_cde_id")
    .withColumn("Current_Status",
        when(col("Current_Status") == "Current Active CC Member", col("snapshot_plan_id"))
        .otherwise(col("Current_Status"))
    )
    .drop("snapshot_plan_id")
)

In [0]:
# =============================================================================
# Add CineClub join date for active CC members
# Source: Customer_Created_Date CSV (customers.id = cc_user_profile_id)
# =============================================================================

# Read join-date CSV
df_join_dates = (
    spark.read.option("header", True)
    .csv("/Volumes/data_exploration_hub/marketing_hub/offline_data/Customer_Created_Date(8th_Aug).csv")
    .select(
        F.expr("try_cast(`customers.id` as BIGINT)").alias("cc_user_profile_id"),
        F.to_timestamp(col("`customers.created_at`"), "M/d/yyyy H:mm").alias("cineclub_join_date")
    )
    .filter(col("cc_user_profile_id").isNotNull())
)

# Map cc_user_profile_id → cde_id via snapshot
df_cc_join = (
    df_snapshot
    .filter(col("snapshot_date_id") == latest_snapshot)
    .filter(col("renewal_status").like("Active%"))
    .select(col("cde_id").cast("int").alias("join_cde_id"),
            F.expr("try_cast(cc_user_profile_id as BIGINT)").alias("cc_user_profile_id"))
    .filter(col("cc_user_profile_id").isNotNull())
    .dropDuplicates(["join_cde_id"])
    .join(df_join_dates, on="cc_user_profile_id", how="inner")
    .select("join_cde_id", "cineclub_join_date")
)

# Attach to df_final (only active members will get a date; others stay null)
# Drop pre-existing column to allow idempotent re-runs
if "cineclub_join_date" in df_final.columns:
    df_final = df_final.drop("cineclub_join_date")

df_final = (
    df_final
    .join(broadcast(df_cc_join), col("CDE_ID") == col("join_cde_id"), "left")
    .drop("join_cde_id")
    .dropDuplicates()

)

In [0]:
df_final.display()

In [0]:
# =============================================================================
# CELL 7b: FINAL OUTPUT SUMMARY
# =============================================================================
# Never CC Members were already filtered to P12M visitors in Cell 7.
# Current Active CC and Former CC Members are included in full.
# =============================================================================

print("FINAL OUTPUT SUMMARY")
print("=" * 60)
df_final.groupBy("Current_Status").agg(
    F.count("*").alias("count"),
    F.round(F.avg("you_have_saved"), 2).alias("avg_you_have_saved"),
    F.round(F.avg("you_could_have_saved"), 2).alias("avg_you_could_have_saved")
).show(truncate=False)

print(f"Total rows: {df_final.count():,}")

In [0]:
# =============================================================================
# CELL 8: WRITE FINAL TABLE TO DELTA
# =============================================================================

table_name = "data_exploration_hub.marketing_hub.cc_lifetime_savings_sendout"

(df_final
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(table_name)
)

print(f"Delta table written to: {table_name}")
print(f"Rows exported: {df_final.count():,}")

In [0]:
# =============================================================================
# TOP CINECLUB MEMBER - COMPREHENSIVE DETAILED STATS
# =============================================================================
# Pulls personalized metrics for the #1 saver among current active CC members.
# Shows TOTAL lifetime metrics AND as-CC-member breakdown.
# Joined CineClub date will be manually provided.
#
# Logic reconciled with upstream Cell 4 savings + Tim Horton's incrementality
# notebook patterns (visit_key, title_id, LC_Location_Type_Description).
# =============================================================================

# --- Step 1: Identify top saver from df_cc_savings ---
top_row = df_cc_savings.orderBy(col("total_savings").desc()).first()
top_cde_id = top_row["CDE_ID"]
total_savings = top_row["total_savings"]
ticket_savings = top_row["ticket_savings"]
concession_savings = top_row["concession_savings"]
gaming_savings = top_row["gaming_savings"]
online_booking_savings = top_row["online_booking_savings"]

# --- Step 2: Pull ALL transactions for this member (CC + non-CC) since Aug 2021 ---
df_all_txns = df_transactions.filter(
    (col("CDE_ID") == top_cde_id) & (col("TransactionDateid") >= 20210801)
)

# --- Step 3: Enrich with dimensions ---
# Film dimension -> title_id for unique titles
df_film = (
    spark.table("cpx_dataanalytics_gold.customer360.cpx_d_film")
    .select("film_id", "title_id")
    .dropDuplicates(["film_id"])
)

# Location dimension -> location type (Theatre vs LBE) + name
df_location = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_location").select(
    F.col("LocationID").alias("loc_id"),
    "Location_Name",
    "LC_Location_Type_Description"
)

df_enriched = (
    df_all_txns
    .join(df_film, on="film_id", how="left")
    .join(df_location, df_all_txns["LocationId"] == df_location["loc_id"], how="left")
    .drop("loc_id")
)

# Visit key = unique date + location combination (same pattern as Tim Horton's notebook)
visit_key = F.concat_ws("|", F.col("VisitDateId").cast("string"), F.col("LocationId").cast("string"))

# --- Step 4: Compute comprehensive metrics ---

# -- 4a. UNIQUE TITLES WATCHED --
unique_titles_total = df_enriched.filter(col("TicketFLAG") == "Yes").select("title_id").distinct().count()
unique_titles_cc = df_enriched.filter(
    (col("TicketFLAG") == "Yes") & (col("CineClubFLAG") == "Yes")
).select("title_id").distinct().count()

# -- 4b. TOTAL THEATRE VISITS (location type like '%Theatre%') --
theatre_visits_total = df_enriched.filter(
    col("LC_Location_Type_Description").like("%Theatre%")
).select(visit_key.alias("vk")).distinct().count()

theatre_visits_cc = df_enriched.filter(
    (col("LC_Location_Type_Description").like("%Theatre%")) & (col("CineClubFLAG") == "Yes")
).select(visit_key.alias("vk")).distinct().count()

# -- 4c. CONCESSION VISITS AT THEATRE --
concession_theatre_visits_total = df_enriched.filter(
    (col("ConcessionFLAG") == "Yes") &
    (col("LC_Location_Type_Description").like("%Theatre%"))
).select(visit_key.alias("vk")).distinct().count()

concession_theatre_visits_cc = df_enriched.filter(
    (col("ConcessionFLAG") == "Yes") &
    (col("CineClubFLAG") == "Yes") &
    (col("LC_Location_Type_Description").like("%Theatre%"))
).select(visit_key.alias("vk")).distinct().count()

# -- 4d. LBE VISITS (RecRoomFLAG or location type like '%Restaurant%') --
lbe_visits_total = df_enriched.filter(
    (col("RecRoomFLAG") == "Yes") |
    (col("LC_Location_Type_Description").like("%Restaurant%"))
).select(visit_key.alias("vk")).distinct().count()

lbe_visits_cc = df_enriched.filter(
    ((col("RecRoomFLAG") == "Yes") | (col("LC_Location_Type_Description").like("%Restaurant%"))) &
    (col("CineClubFLAG") == "Yes")
).select(visit_key.alias("vk")).distinct().count()

# -- 4e. TICKETS PURCHASED --
tickets_total = int(df_enriched.filter(
    col("TicketFLAG") == "Yes"
).agg(F.sum("Quantity")).first()[0] or 0)

tickets_cc = int(df_enriched.filter(
    (col("TicketFLAG") == "Yes") & (col("CineClubFLAG") == "Yes")
).agg(F.sum("Quantity")).first()[0] or 0)

# -- 4f. CC TICKET BREAKDOWN: free tickets (type 6) vs member-priced (type 5) --
member_priced_tickets = int(df_enriched.filter(
    (col("TicketFLAG") == "Yes") & (col("CineClub_Discount_Type_ID") == 5)
).agg(F.sum("Quantity")).first()[0] or 0)

free_tickets_redeemed = int(df_enriched.filter(
    (col("TicketFLAG") == "Yes") & (col("CineClub_Discount_Type_ID") == 6)
).agg(F.sum("Quantity")).first()[0] or 0)

# -- 4g. CONCESSION ITEMS PURCHASED (CC only, consistent with savings logic) --
concession_items_cc = int(df_enriched.filter(
    (col("ConcessionFLAG") == "Yes") & (col("CineClubFLAG") == "Yes")
).agg(F.sum("Quantity")).first()[0] or 0)

concession_items_total = int(df_enriched.filter(
    col("ConcessionFLAG") == "Yes"
).agg(F.sum("Quantity")).first()[0] or 0)

# -- 4h. MODE OF LOCATION (most frequent theatre location across ALL visits) --
mode_loc_row = (
    df_enriched
    .filter(col("LC_Location_Type_Description").like("%Theatre%"))
    .groupBy("LocationId")
    .agg(F.countDistinct(visit_key).alias("visit_count"))
    .orderBy(col("visit_count").desc())
    .first()
)
if mode_loc_row:
    mode_loc_id = mode_loc_row["LocationId"]
    mode_loc_visits = mode_loc_row["visit_count"]
    loc_name_row = df_location.filter(col("loc_id") == mode_loc_id).select("Location_Name").first()
    mode_location_name = loc_name_row["Location_Name"] if loc_name_row else f"Location ID {mode_loc_id}"
else:
    mode_location_name = "N/A"
    mode_loc_visits = 0

# --- Step 5: Print Comprehensive Summary ---
# Pre-format dollar amounts to avoid f-string nesting issues
fmt_ticket = f"${ticket_savings:,.2f}"
fmt_concession = f"${concession_savings:,.2f}"
fmt_gaming = f"${gaming_savings:,.2f}"
fmt_online = f"${online_booking_savings:,.2f}"
fmt_total = f"${total_savings:,.2f}"

print("=" * 70)
print(f"  TOP CINECLUB MEMBER - COMPREHENSIVE STATS")
print(f"  CDE_ID: {top_cde_id}")
print("=" * 70)

header = f"{'METRIC':<50} {'TOTAL':>8} {'AS CC':>8}"
print(f"\n{header}")
print("-" * 70)
print(f"{'Unique titles watched':<50} {unique_titles_total:>8} {unique_titles_cc:>8}")
print(f"{'Theatre visits':<50} {theatre_visits_total:>8} {theatre_visits_cc:>8}")
print(f"{'Concession visits at theatre':<50} {concession_theatre_visits_total:>8} {concession_theatre_visits_cc:>8}")
print(f"{'LBE visits (Rec Room / Restaurant)':<50} {lbe_visits_total:>8} {lbe_visits_cc:>8}")
print(f"{'Tickets purchased':<50} {tickets_total:>8} {tickets_cc:>8}")
print(f"{'Concession items purchased':<50} {concession_items_total:>8} {concession_items_cc:>8}")

print("\nCC TICKET BREAKDOWN")
print("-" * 70)
print(f"{'  Free monthly ticket credits used (type 6)':<50} {free_tickets_redeemed:>8}")
print(f"{'  Member-priced tickets (type 5)':<50} {member_priced_tickets:>8}")

print("\nSAVINGS BREAKDOWN")
print("-" * 70)
print(f"{'  Ticket savings (GA price - $9.99/ticket)':<50} {fmt_ticket:>10}")
print(f"{'  Concession savings (20% discount, type 2)':<50} {fmt_concession:>10}")
print(f"{'  Gaming/LBE savings (20% discount, type 3)':<50} {fmt_gaming:>10}")
print(f"{'  Online booking fee waived ($1/visit)':<50} {fmt_online:>10}")
print(f"{'  TOTAL SAVINGS':<50} {fmt_total:>10}")

print("\nOTHER")
print("-" * 70)
print("  Joined CineClub on: [DATE - TO BE PROVIDED MANUALLY]")
print(f"  Top CineClub theatre (mode): {mode_location_name} ({mode_loc_visits} visits)")
print("=" * 70)

In [0]:
cc_user_profile_id = (
    df_snapshot
    .filter(col("snapshot_date_id") == latest_snapshot)
    .filter(col("cde_id") == top_cde_id)
    .select("cc_user_profile_id")
    .first()
)
cc_user_profile_id = cc_user_profile_id["cc_user_profile_id"] if cc_user_profile_id else "N/A"

In [0]:
print(f"  cc_user_profile_id: {cc_user_profile_id}")